# Lab Session 1: From the Unstructured Web to Structured Entities

**Authors:** Daniel ORDUYREY & Yanis NAITKACI

**Objective:** Build an end-to-end pipeline that (1) crawls and cleans raw web pages, (2) extracts Named Entities, and (3) identifies semantic relations between those entities to populate a Knowledge Graph.

## Phase 1: Web Crawling & Cleaning

### 1.1 Source Selection

We target NASA history pages as seed URLs. These are content-rich, publicly accessible, and highly relevant to our space-exploration knowledge graph. All URLs come from the same domain family, making `robots.txt` compliance straightforward.

In [1]:
SEEDS = [
    "https://history.nasa.gov/SP-4225/documentation/mars/mars.htm",
    "https://www.nasa.gov/missions/artemis/artemis-i-map/",
    "https://history.nasa.gov/SP-4212/ch10.htm",
    "https://history.nasa.gov/SP-4201/ch9.htm",
    "https://history.nasa.gov/SP-4214/ch2.htm"
]

### 1.2 The Extraction Pipeline

We use **trafilatura** to strip HTML noise and extract the main textual content from each page — it is specifically designed to ignore boilerplate like navigation menus and footers. Before fetching, we check the site's `robots.txt` to ensure compliance with its crawling policy. Pages with fewer than 500 words are discarded: short pages are unlikely to carry enough context for meaningful entity extraction. Compliant pages are saved to `crawler_output.jsonl` (one JSON object per line) for reproducible downstream processing.

In [2]:
import httpx
import trafilatura
import json
import time
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

# Simulate a browser user-agent to avoid trivial blocking
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AI-Education-Project-Bot"}

def is_allowed(url):
    """Check robots.txt compliance before fetching a URL."""
    parsed = urlparse(url)
    rp = RobotFileParser()
    rp.set_url(f"{parsed.scheme}://{parsed.netloc}/robots.txt")
    try:
        rp.read()
        return rp.can_fetch("*", url)
    except:
        return True  # Allow if robots.txt is unreachable

def crawl_and_save(urls, output_file="crawler_output.jsonl"):
    """
    Fetch pages, extract text with trafilatura, and save pages
    with >= 500 words to a JSONL file (one valid JSON object per line).
    """
    results_count = 0
    with open(output_file, "w", encoding="utf-8") as f:
        for url in urls:
            if not is_allowed(url):
                print(f"Skipping (robots.txt disallows): {url}")
                continue

            print(f"Processing: {url}")
            try:
                response = httpx.get(url, headers=HEADERS, follow_redirects=True, timeout=10.0)
                # trafilatura strips HTML boilerplate, keeping only the main content
                text = trafilatura.extract(response.text, include_comments=False)

                if text:
                    word_count = len(text.split())
                    if word_count >= 500:  # Challenge threshold: discard short pages
                        record = {"url": url, "text": text}
                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        print(f"  -> Saved ({word_count} words)")
                        results_count += 1
                    else:
                        print(f"  -> Skipped: too short ({word_count} words)")
                else:
                    print("  -> Content not extractable by trafilatura")
            except Exception as e:
                print(f"  -> Request error: {e}")

            time.sleep(2)  # Polite crawling delay

    print(f"\nDone. {results_count} documents saved to '{output_file}'.")

crawl_and_save(SEEDS)

Processing: https://history.nasa.gov/SP-4225/documentation/mars/mars.htm
  -> Saved (936 words)
Processing: https://www.nasa.gov/missions/artemis/artemis-i-map/
  -> Saved (944 words)
Processing: https://history.nasa.gov/SP-4212/ch10.htm
  -> Saved (944 words)
Processing: https://history.nasa.gov/SP-4201/ch9.htm
  -> Saved (944 words)
Processing: https://history.nasa.gov/SP-4214/ch2.htm
  -> Saved (944 words)

Done. 5 documents saved to 'crawler_output.jsonl'.


## Phase 2: Information Extraction

### 2.1 Named Entity Recognition (NER)

We load spaCy's transformer-based model `en_core_web_trf`, which offers the best accuracy for entity recognition at the cost of higher compute. We filter entities to four types relevant to our knowledge graph: **PERSON**, **ORG**, **GPE**, and **DATE**. An additional heuristic — requiring the entity text to start with a capital letter — removes common nouns that spaCy may incorrectly tag (e.g., "mission" labelled as ORG). Each extracted entity is saved with its **sentence context**, which is critical for later ambiguity analysis. Results are deduplicated and stored in `extracted_knowledge.csv`.

In [3]:
import spacy
import pandas as pd
import json

# Load the transformer model for highest-accuracy NER
try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    print("Model not found. Run: python -m spacy download en_core_web_trf")

def extract_entities(input_file="crawler_output.jsonl", output_file="extracted_knowledge.csv"):
    """
    Extract named entities (PERSON, ORG, GPE, DATE) from crawled documents.
    Filters common-noun false positives by requiring an initial capital letter.
    Saves entity text, type, source URL, and sentence context.
    """
    entities_data = []
    allowed_labels = ["PERSON", "ORG", "GPE", "DATE"]

    try:
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    doc = nlp(data["text"])

                    for sent in doc.sents:
                        for ent in sent.ents:
                            # Filter 1: only target labels
                            if ent.label_ not in allowed_labels:
                                continue
                            # Filter 2: remove common-noun false positives
                            #           (real named entities start with a capital letter)
                            if not ent.text[0].isupper():
                                continue

                            entities_data.append({
                                "entity":   ent.text.strip(),
                                "type":     ent.label_,
                                "url":      data["url"],
                                "sentence": sent.text.strip()  # context for ambiguity analysis
                            })
                except json.JSONDecodeError:
                    print("JSON decode error on one line — skipping.")
    except FileNotFoundError:
        print(f"Error: '{input_file}' not found. Did Phase 1 run successfully?")
        return pd.DataFrame()

    df = pd.DataFrame(entities_data).drop_duplicates()
    df.to_csv(output_file, index=False, encoding="utf-8")
    return df

df_entities = extract_entities()
if not df_entities.empty:
    print(f"Extraction complete: {len(df_entities)} unique entities saved to 'extracted_knowledge.csv'.")
    print(df_entities.head(10))

Extraction complete: 142 unique entities saved to 'extracted_knowledge.csv'.
                         entity    type  \
0                         NASA+     ORG   
1                          NASA     ORG   
4                       Artemis     ORG   
5                          Mars     ORG   
6   International Space Station     ORG   
7                          NASA     ORG   
9                          NASA     ORG   
10                        Daily    DATE   
13                   James Webb  PERSON   
14                          Air     ORG   

                                                  url  \
0   https://history.nasa.gov/SP-4225/documentation...   
1   https://history.nasa.gov/SP-4225/documentation...   
4   https://history.nasa.gov/SP-4225/documentation...   
5   https://history.nasa.gov/SP-4225/documentation...   
6   https://history.nasa.gov/SP-4225/documentation...   
7   https://history.nasa.gov/SP-4225/documentation...   
9   https://history.nasa.gov/SP-4225/documentation

### 2.2 Relation Extraction

To populate Knowledge Graph edges, we need structured triplets of the form **(Subject → Verb → Object)**. The key constraint is that **both the subject and the object must be Named Entities** recognised by spaCy — otherwise we would generate noisy triplets linking common words like "program" or "mission".

**Strategy A — Syntactic dependency parsing:** For each sentence, we identify its named entities, then iterate over verbs. For each verb, we look for children tagged as `nsubj`/`nsubjpass` (grammatical subjects) and `dobj`/`pobj`/`attr` (grammatical objects). A triplet is emitted only when both the subject token and the object token can be mapped to a known Named Entity.

**Strategy B — Co-occurrence fallback:** If a sentence contains several named entities but no verb links them syntactically, we create a generic `co-occurs_with` relation between each pair. This preserves contextual information that would otherwise be lost.

In [4]:
import spacy
import pandas as pd
import json
from itertools import combinations

try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    print("Model not found. Run: python -m spacy download en_core_web_trf")

ALLOWED_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

def _build_token_to_entity_map(sent_ents):
    """
    Build a mapping from every token index inside an entity span
    to that entity's text, so we can quickly check if a syntactic
    subject/object token belongs to a Named Entity.
    """
    mapping = {}
    for ent in sent_ents:
        for tok in ent:
            mapping[tok.i] = ent.text.strip()
    return mapping

def extract_relations(input_file="crawler_output.jsonl", output_file="extracted_relations.csv"):
    """
    Extract (Subject, Relation, Object) triplets where both Subject and Object
    are Named Entities. Falls back to co-occurs_with when no direct verb link
    is found between entities in the same sentence.
    """
    relations_data = []

    try:
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    doc = nlp(data["text"])

                    for sent in doc.sents:
                        # Step 1: collect valid Named Entities in this sentence
                        sent_ents = [
                            ent for ent in sent.ents
                            if ent.label_ in ALLOWED_LABELS and ent.text[0].isupper()
                        ]
                        if len(sent_ents) < 2:
                            continue  # Need at least 2 entities for a relation

                        # Step 2: build token → entity name lookup
                        tok2ent = _build_token_to_entity_map(sent_ents)

                        # Step 3 — Strategy A: verb-anchored triplets
                        strategy_a_fired = False
                        for token in sent:
                            if token.pos_ != "VERB":
                                continue

                            subj_tokens = [c for c in token.children
                                           if c.dep_ in ("nsubj", "nsubjpass")]
                            obj_tokens  = [c for c in token.children
                                           if c.dep_ in ("dobj", "pobj", "attr")]

                            for s in subj_tokens:
                                for o in obj_tokens:
                                    # Only emit if BOTH endpoints are Named Entities
                                    if s.i in tok2ent and o.i in tok2ent:
                                        relations_data.append({
                                            "Subject":  tok2ent[s.i],
                                            "Relation": token.lemma_,
                                            "Object":   tok2ent[o.i],
                                            "URL":      data["url"],
                                            "Strategy": "A_syntactic"
                                        })
                                        strategy_a_fired = True

                        # Step 4 — Strategy B: co-occurrence fallback
                        if not strategy_a_fired:
                            for ent_a, ent_b in combinations(sent_ents, 2):
                                relations_data.append({
                                    "Subject":  ent_a.text.strip(),
                                    "Relation": "co-occurs_with",
                                    "Object":   ent_b.text.strip(),
                                    "URL":      data["url"],
                                    "Strategy": "B_cooccurrence"
                                })

                except json.JSONDecodeError:
                    continue

    except FileNotFoundError:
        print(f"Error: '{input_file}' not found. Did Phase 1 run successfully?")
        return pd.DataFrame()

    df = pd.DataFrame(relations_data)
    if not df.empty:
        df.to_csv(output_file, index=False, encoding="utf-8")
    return df

df_relations = extract_relations()
if not df_relations.empty:
    print(f"Extraction complete: {len(df_relations)} triplets saved to 'extracted_relations.csv'.")
    print(df_relations.head(10))
    print("\nStrategy breakdown:")
    print(df_relations["Strategy"].value_counts())
else:
    print("No relations found.")

Extraction complete: 615 triplets saved to 'extracted_relations.csv'.
  Subject        Relation                       Object  \
0   NASA+  co-occurs_with                         NASA   
1   NASA+  co-occurs_with                         NASA   
2   NASA+  co-occurs_with                         NASA   
3   NASA+  co-occurs_with                      Artemis   
4   NASA+  co-occurs_with                         Mars   
5   NASA+  co-occurs_with  International Space Station   
6    NASA  co-occurs_with                         NASA   
7    NASA  co-occurs_with                         NASA   
8    NASA  co-occurs_with                      Artemis   
9    NASA  co-occurs_with                         Mars   

                                                 URL        Strategy  
0  https://history.nasa.gov/SP-4225/documentation...  B_cooccurrence  
1  https://history.nasa.gov/SP-4225/documentation...  B_cooccurrence  
2  https://history.nasa.gov/SP-4225/documentation...  B_cooccurrence  
3  http

## Lab Report

---

### 1. Methodology

**Phase 1 — Web Crawling & Cleaning:** Pages are fetched with `httpx` after verifying the target domain's `robots.txt` with Python's built-in `RobotFileParser`. Raw HTML is passed directly to **trafilatura**, which identifies the main content zone and discards navigation bars, footers, and ads. Pages with fewer than 500 words after extraction are discarded; surviving pages are serialised as a single JSON object `{"url": ..., "text": ...}` and appended to `crawler_output.jsonl`. A 2-second delay between requests ensures polite crawling.

**Phase 2.1 — Named Entity Recognition:** spaCy's `en_core_web_trf` (transformer-based) model processes each document. Only entities labelled **PERSON**, **ORG**, **GPE**, or **DATE** are retained; a capital-letter heuristic (`entity.text[0].isupper()`) filters common-noun false positives. Each kept entity is stored with the full sentence it appears in, producing the `extracted_knowledge.csv` file.

**Phase 2.2 — Relation Extraction:** Relations are extracted at the sentence level using dependency parsing. For every verb in a sentence, we look for syntactic subject children (`nsubj`, `nsubjpass`) and object children (`dobj`, `pobj`, `attr`). A `_build_token_to_entity_map` function maps every token inside a Named Entity span to that entity's canonical text; a triplet **(S → Verb → O)** is emitted only when both the subject token and the object token appear in this map — i.e., only when both are recognised Named Entities. When no verb-anchored triplet can be formed but a sentence contains two or more entities, a fallback `co-occurs_with` relation is generated between each pair (Strategy B), preserving contextual co-presence information.

---

### 2. Ambiguity Analysis

In NLP, entity ambiguity occurs when surface form does not uniquely determine the referent. Three cases are encountered in this corpus:

* **Case 1 — Semantic polysemy (Location vs. Organisation):** The entity **"Mars"** was tagged as `ORG` rather than `GPE`/`LOC` in several sentences. Because the texts often use personifying constructions ("Mars will host", "Mars requires"), the model infers agency, confusing the planet with an organisational actor.

* **Case 2 — Named entity scope (Person vs. Instrument):** **"James Webb"** or **"Webb"** is consistently labelled `PERSON`. In the modern space-exploration context, however, this token almost always refers to the James Webb Space Telescope, which should ideally carry the label `FAC` or `PRODUCT`. The model cannot resolve this without additional world knowledge about the document's time frame.

* **Case 3 — Tokenisation and entity resolution:** **"NASA"** and **"NASA+"** are extracted as distinct entities. The `+` may be stray punctuation, or it may genuinely refer to the agency's streaming service. Without an *entity resolution* step that merges surface variants into a single canonical node, the resulting Knowledge Graph will have fragmented entries for what should be one entity.

---

### 3. Scaling Reflection: From 10 to 10,000 Pages

* **Crawler architecture — asynchronous fetching:** The current sequential loop with `time.sleep()` would require days to process 10,000 pages. A production crawler must be rewritten with an asynchronous framework such as **Scrapy** or Python's `asyncio` + `aiohttp`, enabling concurrent HTTP requests. A message queue (RabbitMQ, Redis) would manage retries and enforce per-domain rate limits without blocking the main loop.

* **NLP resource management — batching and hardware:** `en_core_web_trf` is computationally heavy. At scale, texts must be fed to the model in batches via `nlp.pipe(texts, batch_size=32)` instead of one document at a time. GPU acceleration (CUDA) is practically mandatory to keep inference times tractable; a single CPU core would create a severe bottleneck.

* **Storage and persistence:** In-memory list accumulation and flat CSV exports break at scale. Extracted entities and triplets should be streamed directly into a **graph database** (e.g., Neo4j) to allow efficient graph queries, or serialised to columnar formats (Apache Parquet) for batch analytics. This also enables incremental updates without reprocessing the entire corpus.